# Heatmap Regressor Evaluation

Loads a checkpoint produced by `scripts/train_demo.py` and inspects its predictions on the held-out validation split. Sections:
1. Load checkpoint + rebuild val dataset
2. Aggregate metrics on val (MSE, Spearman, R²)
3. Per-task breakdown
4. Predicted-vs-target panels (browseable)
5. Per-entity score comparison + scatter
6. Worst-case + best-case examples

In [ ]:
import os, sys, json, glob
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import mujoco
from scipy.stats import spearmanr

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)

from planner.risk.dataset import HeatmapDataset, DatasetStats, split_traj_keys
from planner.risk.model import HeatmapMLP
from planner.risk.spatial import load_grid, entity_footprints, integrate_per_entity

# --- Pick the most recent run, or set CHECKPOINT manually ---
runs = sorted(glob.glob('runs/heatmap_*/best.pt'),
              key=lambda p: os.path.getmtime(p), reverse=True)
CHECKPOINT = runs[0] if runs else None
print(f'using checkpoint: {CHECKPOINT}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

## 1. Load model + rebuild val split

In [ ]:
ckpt = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
scene = ckpt['scene']
grid_shape = tuple(ckpt['grid_shape'])
ckpt_args = ckpt['args']
stats = DatasetStats.from_dict(ckpt['stats'])
history = ckpt['history']
print(f'scene={scene}  grid_shape={grid_shape}  best epoch={ckpt["epoch"]}')
print(f'val_loss={min(history["val_loss"]):.4f}   '
      f'val_mse_destd={min(history["val_mse_destd"]):.3f}   '
      f'best ρ={max(history["val_spearman"]):.3f}')

# Rebuild the EXACT val split used during training
DATASET_ROOT = ckpt_args.get('dataset', 'datasets/v10')
SCENES_DIR   = ckpt_args.get('scenes_dir', 'scenes')
SEED         = int(ckpt_args.get('seed', 0))
VAL_FRAC     = float(ckpt_args.get('val_frac', 0.10))

full = HeatmapDataset(DATASET_ROOT, scene)
train_keys, val_keys = split_traj_keys(full.traj_keys, val_frac=VAL_FRAC, seed=SEED)
val_ds = HeatmapDataset(DATASET_ROOT, scene, traj_keys=val_keys, stats=stats)
print(f'val configs: {len(val_ds)}  (over {len(val_keys)} held-out (task,traj) keys)')

model = HeatmapMLP(in_dim=HeatmapDataset.INPUT_DIM, grid_shape=grid_shape).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'model: {n_params/1e6:.2f}M params, loaded from epoch {ckpt["epoch"]}')

In [ ]:
# Sanity check: replot the training history.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
ep = np.arange(1, len(history['train_loss']) + 1)
axes[0].plot(ep, history['train_loss'], label='train')
axes[0].plot(ep, history['val_loss'], label='val')
axes[0].set(title='loss (standardised MSE)', xlabel='epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, history['val_mse_destd'], color='crimson')
axes[1].set(title='val MSE (original units)', xlabel='epoch'); axes[1].grid(alpha=0.3)
axes[2].plot(ep, history['val_spearman'], color='forestgreen')
axes[2].set(title='Spearman ρ (obstacle scores)', xlabel='epoch'); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Run val once, cache predictions

Stores predictions, targets, per-sample MSE so the rest of the notebook is interactive without re-running the model.

In [ ]:
from torch.utils.data import DataLoader
loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2)

y_mean_t = torch.from_numpy(stats.y_mean).to(DEVICE)
y_std_t  = torch.from_numpy(stats.y_std).to(DEVICE)

all_pred = np.zeros((len(val_ds), *grid_shape), dtype=np.float32)
all_targ = np.zeros((len(val_ds), *grid_shape), dtype=np.float32)
all_mse  = np.zeros(len(val_ds), dtype=np.float32)   # de-standardised per-sample MSE
i0 = 0
with torch.no_grad():
    for x, y, _ in loader:
        x = x.to(DEVICE); y = y.to(DEVICE)
        p = model(x)
        # de-standardise
        p_d = p * y_std_t + y_mean_t
        y_d = y * y_std_t + y_mean_t
        n = p_d.shape[0]
        all_pred[i0:i0+n] = p_d.cpu().numpy()
        all_targ[i0:i0+n] = y_d.cpu().numpy()
        all_mse[i0:i0+n]  = ((p_d - y_d) ** 2).mean(dim=(1, 2)).cpu().numpy()
        i0 += n

# Aggregate metrics in original units
mse_overall = float(((all_pred - all_targ) ** 2).mean())
tgt_var = float(all_targ.var())
r2 = 1.0 - mse_overall / tgt_var if tgt_var > 0 else float('nan')
# Baseline: predict the train-mean heatmap for every val sample
baseline_pred = np.broadcast_to(stats.y_mean[None], all_targ.shape)
mse_baseline = float(((baseline_pred - all_targ) ** 2).mean())

print(f'overall val MSE (original units): {mse_overall:.3f}')
print(f'train-mean baseline MSE:          {mse_baseline:.3f}  ->  reduction = '
      f'{(1 - mse_overall/mse_baseline)*100:.1f}%')
print(f'R² (target variance):             {r2:.3f}')
print(f'per-sample MSE: median={np.median(all_mse):.3f}  '
      f'p90={np.percentile(all_mse, 90):.3f}  max={all_mse.max():.3f}')

## 3. Per-task breakdown

How well does the model generalize to each task type? Configs from totally unseen `(task_id, traj_id)` pairs only.

In [ ]:
from collections import defaultdict
by_task = defaultdict(list)
for i, row in enumerate(val_ds.rows):
    by_task[row.task_id].append(i)

rows = []
for task in sorted(by_task):
    idx = np.array(by_task[task])
    mse_t = float(all_mse[idx].mean())
    rows.append((task, len(idx), mse_t, mse_t / mse_baseline))

import pandas as pd
df = pd.DataFrame(rows, columns=['task', 'n', 'val_mse', 'mse_vs_baseline'])
df = df.sort_values('val_mse')
display(df)

fig, ax = plt.subplots(figsize=(9, max(3, 0.35 * len(df))))
ax.barh(df['task'], df['val_mse'], color='crimson', alpha=0.8)
ax.axvline(mse_baseline, color='gray', linestyle='--', label=f'mean-baseline = {mse_baseline:.2f}')
ax.axvline(mse_overall, color='black', linestyle=':', label=f'overall = {mse_overall:.2f}')
ax.set_xlabel('val MSE (original units)'); ax.legend()
ax.invert_yaxis(); ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

## 4. Predicted-vs-target panels (browse N random val samples)

Change `N` and rerun the cell to draw a fresh random sample.

In [ ]:
N = 6
rng = np.random.default_rng()
sel = rng.choice(len(val_ds), size=N, replace=False)

grid = load_grid(Path(DATASET_ROOT) / scene / 'grid.json')
fig, axes = plt.subplots(N, 3, figsize=(11, 2.6 * N))
if N == 1: axes = axes.reshape(1, -1)
for k, idx in enumerate(sel):
    row = val_ds.rows[idx]
    p = all_pred[idx]; t = all_targ[idx]; e = np.abs(p - t)
    vmax = max(p.max(), t.max(), 1e-6)
    for c, (img, title, vm) in enumerate([(t, 'target', vmax), (p, 'pred', vmax),
                                          (e, '|err|', e.max() + 1e-6)]):
        ax = axes[k, c]
        ax.imshow(img, origin='lower', extent=grid.extent, cmap='magma', vmin=0, vmax=vm)
        ax.set_xticks([]); ax.set_yticks([])
        if k == 0: ax.set_title(title)
    axes[k, 0].set_ylabel(f'{row.task_id}\ntraj{row.traj_id}\nMSE={all_mse[idx]:.2f}', fontsize=8)
plt.tight_layout(); plt.show()

## 5. Per-entity integrated scores (predicted vs target)

Integrate predicted and target heatmaps over each entity's X-Y AABB and compare. This is exactly the per-entity score that gets multiplied by severity to compute the planner's safety cost.

In [ ]:
mj_model = mujoco.MjModel.from_xml_path(f'{SCENES_DIR}/{scene}/scene.xml')
fps = entity_footprints(mj_model, grid)
ent_names = list(fps.keys())
print('entities:', ent_names)

# Per-sample integrated score, per entity.
P = np.zeros((len(val_ds), len(ent_names)), dtype=np.float32)
T = np.zeros_like(P)
for i in range(len(val_ds)):
    p_scores = integrate_per_entity(all_pred[i], fps)
    t_scores = integrate_per_entity(all_targ[i], fps)
    for j, n in enumerate(ent_names):
        P[i, j] = p_scores.get(n, 0.0)
        T[i, j] = t_scores.get(n, 0.0)

# Per-entity Spearman + scatter
rho_per_ent = []
for j, n in enumerate(ent_names):
    if P[:, j].std() > 0 and T[:, j].std() > 0:
        rho, _ = spearmanr(P[:, j], T[:, j])
    else:
        rho = float('nan')
    rho_per_ent.append(rho)

for n, r in sorted(zip(ent_names, rho_per_ent), key=lambda kv: -(kv[1] if kv[1]==kv[1] else -2)):
    print(f'  {n:22s}  ρ={r:.3f}')

In [ ]:
ncols = 3
nrows = (len(ent_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows), squeeze=False)
for j, name in enumerate(ent_names):
    ax = axes[j // ncols, j % ncols]
    ax.scatter(T[:, j], P[:, j], s=10, alpha=0.4, color='crimson')
    lo = min(T[:, j].min(), P[:, j].min(), 0)
    hi = max(T[:, j].max(), P[:, j].max(), 1e-6)
    ax.plot([lo, hi], [lo, hi], color='black', linestyle='--', alpha=0.5, lw=1)
    ax.set(title=f'{name}  ρ={rho_per_ent[j]:.3f}', xlabel='target score', ylabel='pred score')
    ax.grid(alpha=0.3)
for j in range(len(ent_names), nrows * ncols):
    axes[j // ncols, j % ncols].axis('off')
plt.tight_layout(); plt.show()

## 6. Worst- and best-case examples

Worst MSE samples expose where the model fails. Best MSE samples confirm what it gets right.

In [ ]:
K = 4
worst = np.argsort(all_mse)[-K:][::-1]
best  = np.argsort(all_mse)[:K]

for label, sel in [('worst', worst), ('best', best)]:
    print(f'--- {label} ---')
    fig, axes = plt.subplots(K, 3, figsize=(11, 2.6 * K))
    if K == 1: axes = axes.reshape(1, -1)
    for k, idx in enumerate(sel):
        row = val_ds.rows[idx]
        p = all_pred[idx]; t = all_targ[idx]; e = np.abs(p - t)
        vmax = max(p.max(), t.max(), 1e-6)
        for c, (img, title, vm) in enumerate([(t, 'target', vmax), (p, 'pred', vmax),
                                              (e, '|err|', e.max() + 1e-6)]):
            ax = axes[k, c]
            ax.imshow(img, origin='lower', extent=grid.extent, cmap='magma', vmin=0, vmax=vm)
            ax.set_xticks([]); ax.set_yticks([])
            if k == 0: ax.set_title(title)
        axes[k, 0].set_ylabel(f'{row.task_id}\ntraj{row.traj_id}\nMSE={all_mse[idx]:.2f}', fontsize=8)
    plt.suptitle(f'{label} {K} val configs by MSE', y=1.005)
    plt.tight_layout(); plt.show()

## 7. Single-config inspector

Pick any val index `i` to render a detailed view: target, pred, error, and the per-entity bar chart for that one config.

In [ ]:
i = 0   # change this to inspect different val configs (0 .. len(val_ds)-1)

row = val_ds.rows[i]
p = all_pred[i]; t = all_targ[i]; e = np.abs(p - t)
vmax = max(p.max(), t.max(), 1e-6)
fig = plt.figure(figsize=(13, 4.2))
for c, (img, title, vm) in enumerate([(t, 'target', vmax), (p, 'pred', vmax),
                                      (e, '|err|', e.max() + 1e-6)]):
    ax = fig.add_subplot(1, 4, c + 1)
    ax.imshow(img, origin='lower', extent=grid.extent, cmap='magma', vmin=0, vmax=vm)
    ax.set_title(title); ax.set_xlabel('X (m)')
    if c == 0: ax.set_ylabel('Y (m)')
ax = fig.add_subplot(1, 4, 4)
items = sorted(zip(ent_names, T[i], P[i]), key=lambda x: -max(x[1], x[2]))
names = [n.replace('obstacle_', '').replace('simple_', '') for n, _, _ in items]
y_pos = np.arange(len(names))
ax.barh(y_pos - 0.2, [v[1] for v in items], height=0.4, label='target', color='gray')
ax.barh(y_pos + 0.2, [v[2] for v in items], height=0.4, label='pred',   color='crimson', alpha=0.85)
ax.set_yticks(y_pos); ax.set_yticklabels(names, fontsize=8)
ax.invert_yaxis(); ax.set_xlabel('integrated score'); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='x')
plt.suptitle(f'val[{i}]  {row.task_id}/traj{row.traj_id}  MSE={all_mse[i]:.3f}', y=1.02)
plt.tight_layout(); plt.show()

## 8. Project predicted heatmap onto front_cam

Take the model's predicted 2D contact density (in world X-Y at the table-top z),
treat each grid cell as a point at `(x_center, y_center, table_z)`, project to
the front_cam via `ContactProjector`, and overlay the predicted intensity on the
trial's `pre_rgb`. Compare side-by-side with the target heatmap projected the
same way to see *where* the model thinks the contacts will land in image space.


In [ ]:
from planner.experiments.data_capture import ContactProjector
from planner.risk.spatial import SCENE_TABLE_Z

# Cache one projector per scene (one MuJoCo model per scene).
_proj_cache = {}
def _projector_for(scene_name, width=640, height=480):
    if scene_name not in _proj_cache:
        m = mujoco.MjModel.from_xml_path(f'{SCENES_DIR}/{scene_name}/scene.xml')
        _proj_cache[scene_name] = (
            m,
            ContactProjector(m, width=width, height=height, camera_name='front_cam'),
        )
    return _proj_cache[scene_name]

# Build the 3D world-frame point per grid cell (cell centre @ table-top z).
table_z = SCENE_TABLE_Z[scene]
bin_m = grid.bin_cm / 100.0
yy, xx = np.mgrid[:grid.ny, :grid.nx]
cell_world = np.stack([
    grid.x_min + (xx + 0.5) * bin_m,
    grid.y_min + (yy + 0.5) * bin_m,
    np.full_like(xx, table_z, dtype=np.float64),
], axis=-1).reshape(-1, 3)            # (ny*nx, 3)
print(f'cell-centre cloud: {cell_world.shape}  table_z={table_z}')

_, proj = _projector_for(scene)
pixels, depths = proj.project(cell_world)
H, W = 480, 640
visible = (depths > 0) & (pixels[:, 0] >= 0) & (pixels[:, 0] < W) \
                       & (pixels[:, 1] >= 0) & (pixels[:, 1] < H)
print(f'cells projecting into front_cam frame: {int(visible.sum())} / {len(visible)}')


In [ ]:
def _overlay_heatmap_on_rgb(rgb, intensity_grid, pixels, visible, alpha_max=0.7,
                            sigma_px=2.5):
    """Render a heatmap on top of an RGB image by accumulating each visible
    grid cell's intensity into a 2D image-plane buffer, then alpha-blending."""
    img = rgb.copy()
    H, W = img.shape[:2]
    flat = intensity_grid.reshape(-1)
    # Build an image-plane intensity buffer
    buf = np.zeros((H, W), dtype=np.float32)
    px = pixels[visible].astype(int)
    flat_v = flat[visible]
    # Accumulate (multiple cells can land in the same pixel due to perspective)
    np.add.at(buf, (px[:, 1], px[:, 0]), flat_v)
    # Smooth so adjacent cells blend visually
    from scipy.ndimage import gaussian_filter
    if sigma_px > 0:
        buf = gaussian_filter(buf, sigma=sigma_px)
    if buf.max() > 0:
        a = alpha_max * (buf / buf.max())
    else:
        a = np.zeros_like(buf)
    # Magma colormap on the buffer
    from matplotlib.cm import magma
    rgba = magma(buf / max(buf.max(), 1e-6))[..., :3]
    out = (1 - a[..., None]) * img / 255.0 + a[..., None] * rgba
    return np.clip(out, 0, 1)


# Pick a val index to inspect (you can change this).
i = int(np.argmin(all_mse))   # default: best-fit example; try np.argmax(all_mse) for the worst
row = val_ds.rows[i]
with np.load(row.npz_path) as d:
    rgb = d['pre_rgb']

t = all_targ[i]; p = all_pred[i]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
axes[0].imshow(rgb); axes[0].set_title('pre_rgb (clean)')
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[1].imshow(_overlay_heatmap_on_rgb(rgb, t, pixels, visible))
axes[1].set_title('target overlay')
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[2].imshow(_overlay_heatmap_on_rgb(rgb, p, pixels, visible))
axes[2].set_title('predicted overlay')
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.suptitle(f'val[{i}]  {row.task_id}/traj{row.traj_id}  MSE={all_mse[i]:.3f}', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Browse a small grid of held-out configs: target vs predicted overlay side-by-side.
N = 4
rng = np.random.default_rng()
sel = rng.choice(len(val_ds), size=N, replace=False)

fig, axes = plt.subplots(N, 2, figsize=(11, 3.5 * N))
if N == 1: axes = axes.reshape(1, -1)
for k, idx in enumerate(sel):
    row = val_ds.rows[idx]
    with np.load(row.npz_path) as d:
        rgb = d['pre_rgb']
    axes[k, 0].imshow(_overlay_heatmap_on_rgb(rgb, all_targ[idx], pixels, visible))
    axes[k, 0].set_title(f'target  |  {row.task_id}/traj{row.traj_id}')
    axes[k, 0].set_xticks([]); axes[k, 0].set_yticks([])
    axes[k, 1].imshow(_overlay_heatmap_on_rgb(rgb, all_pred[idx], pixels, visible))
    axes[k, 1].set_title(f'pred (MSE={all_mse[idx]:.2f})')
    axes[k, 1].set_xticks([]); axes[k, 1].set_yticks([])
plt.tight_layout(); plt.show()
